In [1]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
from transformers import AutoTokenizer, AutoModel
import torch
from typing import Literal

PoolingMethod = Literal["mean", "max", "cls"]

class SentenceEmbeddings:
	def __init__(self, model_name: str, device: str | None = None):
		self.device = device if device else ("cuda" if torch.cuda.is_available() else "cpu")
		self.tokenizer = AutoTokenizer.from_pretrained(model_name)
		self.model = AutoModel.from_pretrained(model_name).to(self.device)
		self.model.eval()

	# Mean pooling captura el contenido semántico general promediando todos los word embeddings.
	def mean_pooling(self, model_output, attention_mask):
		# token_embeddings = model_output[0]
		token_embeddings = model_output["last_hidden_state"]
		# Modificar el pooling para tener en cuenta la máscara de atención. Esta máscara es un vector de 0 y 1 que indica qué tokens son reales y cuáles son de relleno. Queremos ignorar los paddings tokens al calcular la media.
		input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
		return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(
			input_mask_expanded.sum(1), min=1e-9
		)
	
	# Max pooling toma los valores máximos de cada dimensión de los word embeddings.
	# Sin embargo, puede hacer que el embedding de la oración sea menos equilbirado porque dos oraciones pueden compartir palabras importante pero diferir en otras, lo que exagera las diferencias del max pooling.
	def max_pooling(self, model_output, attention_mask):
		token_embeddings = model_output["last_hidden_state"]
		input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
		
		# Enmascara los padding tokens con un número negativo muy grande para que se ignoren en el máximo.
		token_embeddings[input_mask_expanded == 0] = -1e9
		return torch.max(token_embeddings, dim=1).values
	
	# [CLS] pooling consiste en usar el embedding del primer token de la secuencia como el embedding de la oración.
	# Este token está diseñador para capturar una representación global de la oración, ya que durante el preentrenamiento
	# se utiliza para tareas como:
	#   - Next Sentence Prediction
	# Sin embargo, el embedding [CLS] no fue entrenado para específicamente para tareas de similitud semántica.
	# Por esto, en muchos casos el mean pooling produce mejores embeddings para comparaciones de oraciones.
	def cls_pooling(self, model_output):
		# Algunos modelos, como BERT, incluyen "pooler_output", que corresponde 
		# al embedding del token [CLS] después de pasar por una capa lineal y una activación tanh.
		if "pooler_output" in model_output:
			return model_output["pooler_output"]
		else:
			return model_output["last_hidden_state"][:, 0]

	def encode(self, sentences: list[str], pooling: PoolingMethod):
		# Modificar la tokenización para aplicar "truncation" (cortar la oración si es más larga que la longitud máxima) y "padding" (agregar [PAD] tokens al final de la oración).
		encoded_input = self.tokenizer(
			sentences,
			padding=True,
			truncation=True,
			return_tensors="pt"
		).to(self.device)

		with torch.no_grad():
			model_output = self.model(**encoded_input)

		match pooling:
			case "mean":
				sentence_embeddings = self.mean_pooling(model_output, encoded_input["attention_mask"])
			case "max":
				sentence_embeddings = self.max_pooling(model_output, encoded_input["attention_mask"])
			case "cls":
				sentence_embeddings = self.cls_pooling(model_output)
				
		print(sentence_embeddings)
		
		sentence_embeddings = torch.nn.functional.normalize(sentence_embeddings, p=2, dim=1)

		print(sentence_embeddings)

		return sentence_embeddings.cpu().numpy()

In [9]:
model = SentenceEmbeddings("dccuchile/bert-base-spanish-wwm-cased")

Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
corpus = [
	"¡Gracias! Si necesito algo, te aviso.",
	"Igualmente, un gusto conocerte *sonríes*.",
	"Ah, sí... ¡hola!"
	]

In [20]:
embeddings = model.encode(corpus, "mean")

print(embeddings)

tensor([[-0.0987, -0.3006, -0.0271,  ..., -0.2055, -0.1701,  0.1825],
        [-0.1721,  0.0399, -0.0626,  ..., -0.3849, -0.1425, -0.0495],
        [-0.5532,  0.5887, -0.4228,  ..., -0.1724, -0.1051,  0.1643]],
       device='cuda:0')
tensor([[-0.0066, -0.0201, -0.0018,  ..., -0.0137, -0.0114,  0.0122],
        [-0.0114,  0.0026, -0.0041,  ..., -0.0255, -0.0094, -0.0033],
        [-0.0357,  0.0379, -0.0273,  ..., -0.0111, -0.0068,  0.0106]],
       device='cuda:0')
[[-0.00659292 -0.02008359 -0.00181256 ... -0.0137302  -0.01136793
   0.01219503]
 [-0.01141302  0.00264333 -0.00414901 ... -0.02552264 -0.00944994
  -0.0032854 ]
 [-0.03565165  0.03794348 -0.02725103 ... -0.01111355 -0.00677521
   0.01058781]]


In [21]:
from app.utils.similarity import cosine_similarity

cosine_similarity(embeddings)

array([[1.        , 0.82642454, 0.83807972],
       [0.82642454, 1.        , 0.83074567],
       [0.83807972, 0.83074567, 1.        ]])